# Custom Object Detection Training — RF-DETR

Train a custom object detector (RF-DETR) on your Roboflow dataset (COCO format)
and download a `.pkl` for the Jetson vision server.

The `.pkl` carries the model as **ONNX**. The Jetson builds its TensorRT engine
from it the first time the model loads (a few minutes, once per model — the
engine is cached on the Jetson). An engine is tied to the GPU that builds it,
so it cannot be built here.

**Important:** Set **Runtime > Change runtime type > GPU** before starting.

## 1. Download Dataset from Roboflow

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("your-workspace").project("your-project")
version = project.version(1)
dataset = version.download("coco")

## 2. Install RF-DETR

In [ ]:
# Pinned: the version the Jetson's OD (dorna_vision/ai.py) is checked against —
# same ONNX inputs/outputs. Bump both together.
!pip install -q "rfdetr[train,onnx]==1.11.2"

## 3. Training Parameters

`MODEL_SIZE` trades speed for accuracy. On the Jetson Orin Nano Super (FP16
TensorRT), `small` runs a 512 x 512 frame in about 20 ms.

In [ ]:
import os
import json
import pickle
from google.colab import files
from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge

# ============================================================
# EDIT THIS SECTION
# ============================================================
MODEL_SIZE = "small"       # "nano" | "small" | "medium" | "base" | "large"
EPOCHS = 100
BATCH_SIZE = 8
GRAD_ACCUM = 2             # effective batch = BATCH_SIZE x GRAD_ACCUM (16 recommended)
LR = 1e-4
RESOLUTION = None          # None = the size's default (nano 384, small 512, medium 576, base 560,
                           # large 704); else a multiple of 32 (base: of 56)

MODELS = {"nano": RFDETRNano, "small": RFDETRSmall, "medium": RFDETRMedium,
          "base": RFDETRBase, "large": RFDETRLarge}
Model = MODELS[MODEL_SIZE]

project_name = project.name.lower().replace(" ", "_")

print(f"Model size:  {MODEL_SIZE}")
print(f"Classes:     {list(project.classes.keys())}")
print(f"Epochs:      {EPOCHS}")
print(f"Batch size:  {BATCH_SIZE} x {GRAD_ACCUM}")
print(f"Resolution:  {RESOLUTION or 'model default'}")

## 4. Verify COCO Dataset Layout

Roboflow's COCO export is RF-DETR's native format — no conversion needed.
This cell just sanity-checks the paths and reports sample counts.

In [ ]:
# Roboflow's COCO export layout:
#   dataset.location/
#     train/_annotations.coco.json + .jpg files
#     valid/_annotations.coco.json + .jpg files
#     test/_annotations.coco.json  + .jpg files

DATA_ROOT = dataset.location

for split in ("train", "valid", "test"):
    ann_path = f"{DATA_ROOT}/{split}/_annotations.coco.json"
    if not os.path.exists(ann_path):
        print(f"  {split}: MISSING  ({ann_path})")
        continue
    with open(ann_path) as f:
        coco = json.load(f)
    print(f"  {split}: {len(coco.get('images', []))} images, {len(coco.get('annotations', []))} annotations")

## 5. Train

In [ ]:
OUTPUT_DIR = "/content/output"

model = Model()
model.train(
    dataset_dir=DATA_ROOT,
    output_dir=OUTPUT_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM,
    lr=LR,
    **({"resolution": RESOLUTION} if RESOLUTION else {}),
)

## 6. Export ONNX

The best checkpoint (by validation mAP) is exported. The check below runs the
ONNX file the way the Jetson does and compares it with RF-DETR's own
prediction on a validation image — a bad export shows here, not on the bench.

In [ ]:
import cv2
import numpy as np
import onnxruntime as ort

best = Model(pretrain_weights=f"{OUTPUT_DIR}/checkpoint_best_total.pth")
image_size = int(best.model_config.resolution)
class_names = list(best.class_names)   # logit slot i is class_names[i]
onnx_path = best.export(output_dir="/content/export", verbose=False)

# the Jetson's preprocessing (dorna_vision/ai.py OD._preprocess)
val_dir = f"{DATA_ROOT}/valid"
sample = sorted(f for f in os.listdir(val_dir) if f.lower().endswith((".jpg", ".jpeg", ".png")))[0]
img = cv2.imread(f"{val_dir}/{sample}")
resized = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB).astype(np.float32)
mean = np.array([0.485, 0.456, 0.406], np.float32) * 255
std = np.array([0.229, 0.224, 0.225], np.float32) * 255
tensor = ((rgb - mean) / std).transpose(2, 0, 1)[None]

dets, logits = ort.InferenceSession(onnx_path).run(["dets", "labels"], {"input": tensor})
scores = 1 / (1 + np.exp(-logits[0]))
q, c = np.unravel_index(scores.argmax(), scores.shape)
ref = best.predict(img[:, :, ::-1].copy(), threshold=0.0)
print(f"ONNX:    {class_names[c]} {scores[q, c]:.3f}")
top = int(np.argmax(ref.confidence))
print(f"RF-DETR: {class_names[int(ref.class_id[top])]} {float(ref.confidence[top]):.3f}")

## 7. Save & Download Pickle

In [ ]:
# Pickle format of the Jetson's OD (dorna_vision/ai.py):
#   { onnx, cls, colors, meta }
with open(onnx_path, "rb") as f:
    onnx_data = f.read()

model_dict = {
    "onnx": onnx_data,
    "cls": class_names,
    "colors": project.colors,
    "meta": {
        "type": "od",
        "arch": "rf-detr",
        "variant": MODEL_SIZE,
        "image_size": image_size,
        "precision": "fp16",   # the Jetson's TensorRT engine; RF-DETR's activations fit FP16
    },
}

pickle_path = f"/content/{project_name}.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(model_dict, f)

print(f"Saved: {pickle_path}")
print(f"Total size: {os.path.getsize(pickle_path) / 1024 / 1024:.1f} MB")
print(f"Model size: {MODEL_SIZE}")
print(f"Classes: {class_names}")
print(f"Image size: {image_size}")

files.download(pickle_path)